# 02 — Bronze: electricity prices

Downloads daily spot prices per price area from elprisetjustnu.se and lands
one JSON file per (price area, date) in the bronze volume.

The API has no query parameters — date and area are path segments, so one call
returns one day for one area. History starts around November 2022.

| Area | Region |
|---|---|
| SE1 | Luleå |
| SE2 | Sundsvall |
| SE3 | Stockholm |
| SE4 | Malmö |

**This is the slowest part of the pipeline.** Roughly 4,300 calls, 25–45
minutes on first run. Files are skipped if already present, so re-runs only
fetch new days and take seconds.

Data is free to use; elprisetjustnu.se asks for attribution.

In [0]:
import os
import time
import datetime
import requests

CATALOG = "axenil_assignment1"
BRONZE_SCHEMA = "bronze"
RAW_DIRECTORY = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/raw/elpris"

PRICE_API_BASE = "https://www.elprisetjustnu.se/api/v1/prices"
PRICE_AREAS = ["SE1", "SE2", "SE3", "SE4"]
REQUEST_HEADERS = {"User-Agent": "nackademin-laddstolpar-axel"}

FIRST_DATE = datetime.date(2022, 11, 1)
LAST_DATE = datetime.date.today()

SECONDS_BETWEEN_REQUESTS = 0.3

In [0]:
def build_price_url(date, price_area):
    return f"{PRICE_API_BASE}/{date.year}/{date.strftime('%m-%d')}_{price_area}.json"


def fetch_price_day(date, price_area):
    directory = f"{RAW_DIRECTORY}/elomrade={price_area}/ar={date.year}"
    file_path = f"{directory}/{date.isoformat()}.json"

    if os.path.exists(file_path):
        return "skipped"

    os.makedirs(directory, exist_ok=True)
    response = requests.get(build_price_url(date, price_area), headers=REQUEST_HEADERS, timeout=60)

    if response.status_code == 404:
        return "missing"
    if response.status_code == 429:
        time.sleep(30)
        return "throttled"
    response.raise_for_status()

    temporary_path = file_path + ".tmp"
    with open(temporary_path, "wb") as output_file:
        output_file.write(response.content)
    os.replace(temporary_path, file_path)
    return "fetched"


result_counts = {"fetched": 0, "skipped": 0, "missing": 0, "throttled": 0}
current_date = FIRST_DATE

while current_date <= LAST_DATE:
    for price_area in PRICE_AREAS:
        result = fetch_price_day(current_date, price_area)
        result_counts[result] += 1
        if result == "fetched":
            time.sleep(SECONDS_BETWEEN_REQUESTS)

    if current_date.day == 1:
        print(current_date, result_counts)
    current_date += datetime.timedelta(days=1)

print("done:", result_counts)

In [0]:
import os
import json
from pyspark.sql import functions
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

TARGET_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.electricity_prices"

PRICE_SCHEMA = StructType([
    StructField("elomrade",     StringType(), False),
    StructField("time_start",   StringType(), False),
    StructField("time_end",     StringType(), False),
    StructField("SEK_per_kWh",  DoubleType(), True),
    StructField("EUR_per_kWh",  DoubleType(), True),
    StructField("EXR",          DoubleType(), True),
    StructField("_source_file", StringType(), False),
])


def as_float(value):
    return None if value is None else float(value)


if spark.catalog.tableExists(TARGET_TABLE):
    already_loaded = {
        row["_source_file"]
        for row in spark.table(TARGET_TABLE).select("_source_file").distinct().collect()
    }
else:
    already_loaded = set()

new_rows = []

for area_directory in sorted(os.listdir(RAW_DIRECTORY)):
    price_area = area_directory.split("=")[1]
    area_path = f"{RAW_DIRECTORY}/{area_directory}"

    for year_directory in sorted(os.listdir(area_path)):
        year_path = f"{area_path}/{year_directory}"

        for file_name in sorted(os.listdir(year_path)):
            if not file_name.endswith(".json"):
                continue

            file_path = f"{year_path}/{file_name}"
            if file_path in already_loaded:
                continue

            with open(file_path, "r", encoding="utf-8") as input_file:
                periods = json.load(input_file)

            for period in periods:
                new_rows.append((
                    price_area,
                    period["time_start"],
                    period["time_end"],
                    as_float(period.get("SEK_per_kWh")),
                    as_float(period.get("EUR_per_kWh")),
                    as_float(period.get("EXR")),
                    file_path,
                ))

new_file_count = len({row[6] for row in new_rows})
print(f"{len(new_rows)} new rows from {new_file_count} files")

if new_rows:
    price_dataframe = (
        spark.createDataFrame(new_rows, PRICE_SCHEMA)
             .withColumn("_ingested_at", functions.current_timestamp())
    )
    price_dataframe.write.mode("append").saveAsTable(TARGET_TABLE)

print("total rows:", spark.table(TARGET_TABLE).count())

In [0]:
electricity_prices = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.electricity_prices")

display(
    electricity_prices.groupBy("elomrade")
          .agg(
              functions.countDistinct("_source_file").alias("amount_days"),
              functions.count("*").alias("amount_hours"),
              functions.min("time_start").alias("earliest"),
              functions.max("time_start").alias("latest"),
          )
          .orderBy("elomrade")
)